# Scraper Studio - Custom Scrapers via SDK

Trigger and fetch results from your custom scrapers built via Bright Data's Scraper Studio (AI Agent, IDE, or templates).

## What You'll Learn
1. Setup and authentication
2. Trigger a custom scraper
3. Fetch results when ready
4. Check job status
5. Multiple inputs

---

## Setup

In [ ]:
#pip install brightdata-sdk

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

API_TOKEN = os.getenv("BRIGHTDATA_API_TOKEN")
if not API_TOKEN:
    raise ValueError("Set BRIGHTDATA_API_TOKEN in .env file")

print(f"API Token: {API_TOKEN[:10]}...{API_TOKEN[-4:]}")
print("Setup complete!")

API Token: 859a9772-4...4362
Setup complete!


## Initialize Client

In [2]:
from brightdata import BrightDataClient

client = BrightDataClient(token=API_TOKEN)
await client.__aenter__()

# Your collector ID from Scraper Studio dashboard
COLLECTOR_ID = "c_mly0sa6x10hshxi8jb"  # Replace with your collector ID

print("Client initialized")

Client initialized


---

## Single URL - Trigger

Trigger the scraper. Returns immediately with a job object containing the `response_id`.

In [3]:
# Trigger - returns immediately
job = await client.scraper_studio.trigger(
    collector=COLLECTOR_ID,
    input={"url": "https://www.sahibinden.com/ilan/emlak-konut-satilik-golden-gate-1287846580/detay"},
)
print(f"Job triggered: {job.response_id}")

Job triggered: d2t1773025017352rhfek0rl202o


## Single URL - Fetch

Try to fetch the result. If not ready yet, re-run this cell.

In [9]:
# Fetch - single attempt, re-run if not ready
try:
    data = await job.fetch()
    print(f"Got {len(data)} record(s)")
    for record in data:
        for key, value in list(record.items())[:5]:
            print(f"  {key}: {value}")
except Exception as e:
    print(f"Not ready yet: {e}\nRe-run this cell in a few seconds.")

Got 1 record(s)
  title: GOLDEN GATE
  price: {'value': 6600000, 'currency': 'TRY', 'symbol': '₺'}
  property_size: 100
  room_count: 3
  building_age: 6-10 arası


---

## Check Job Status

Check the status of a previously triggered job using its job ID (from the Scraper Studio dashboard).

In [11]:
# Check status of a known job
JOB_ID = "j_mly4pzxd1mj4u0gjj8"  # Replace with your job ID

info = await client.scraper_studio.status(job_id=JOB_ID)

print(f"Job ID:       {info.id}")
print(f"Status:       {info.status}")
print(f"Collector:    {info.collector}")
print(f"Inputs:       {info.inputs}")
print(f"Lines:        {info.lines}")
print(f"Success rate: {info.success_rate}")
print(f"Job time:     {info.job_time}ms")

Job ID:       j_mly4pzxd1mj4u0gjj8
Status:       done
Collector:    c_mly0sa6x10hshxi8jb
Inputs:       1
Lines:        1
Success rate: 1
Job time:     106996ms


---

## Multiple Inputs

`run()` accepts a list of inputs, triggers each, polls, and returns combined results.

In [ ]:
urls = [
    {"url": "https://www.sahibinden.com/ilan/emlak-konut-satilik-golden-gate-1287846580/detay"},
    {"url": "https://www.sahibinden.com/ilan/emlak-konut-satilik-golden-gate-1287846581/detay"},
]

multi_data = await client.scraper_studio.run(
    collector=COLLECTOR_ID,
    input=urls,
    timeout=300,
)

print(f"Got {len(multi_data)} total record(s)")
for record in multi_data:
    print(f"  - {record.get('title', 'N/A')}")

---

## Save Results

Save the scraped data to a JSON file.

In [ ]:
import json

with open("scraper_studio_results.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"Saved {len(data)} record(s) to scraper_studio_results.json")

---

## Cleanup

In [ ]:
await client.__aexit__(None, None, None)
print("Client closed.")

---

## Summary

| Method | What it does |
|--------|-------------|
| `client.scraper_studio.run(collector, input)` | Trigger + poll + return data |
| `client.scraper_studio.trigger(collector, input)` | Trigger only, returns job object |
| `job.fetch()` | Single fetch attempt |
| `job.wait_and_fetch(timeout)` | Poll until data arrives |
| `client.scraper_studio.status(job_id)` | Check job status |
| `client.scraper_studio.fetch(response_id)` | Fetch results by response_id |

## Resources

- [Scraper Studio Dashboard](https://brightdata.com/cp/data_collector)
- [API Reference](https://docs.brightdata.com/api-reference/scraper-studio-api/)